# 🤖 KI Trainer Studio - Interaktives Notebook

Dieses Notebook ist dein komplettes KI-Labor. Du kannst hier:
- Daten laden & visualisieren
- Modelle erstellen & konfigurieren  
- Live trainieren mit Plots
- Exportieren (ONNX, PyTorch, Sklearn)

Einfach Zelle für Zelle ausführen (Shift+Enter)

## 1️⃣ Setup & Imports

In [ ]:
# Installation (falls nötig)
# !pip install torch torchvision scikit-learn pandas matplotlib tqdm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from ki_trainer import TrainerConfig, KiTrainer, DataManager

plt.style.use('dark_background')
print("✅ Imports OK")
print(f"Torch CUDA verfügbar: {__import__('torch').cuda.is_available()}")

## 2️⃣ Datensatz laden
Wähle eine Option:

In [ ]:
# OPTION A: Demo Datensatz (Iris / California Housing)
config = TrainerConfig(
    task="classification",  # oder "regression"
    model_type="mlp",
    test_split=0.2
)
data_manager = DataManager(config)
train_loader, test_loader = data_manager.load_demo("iris")  # iris oder california

print(f"Input Dim: {data_manager.input_dim}, Output Dim: {data_manager.output_dim}")

In [ ]:
# OPTION B: Eigene CSV laden
# csv_path = "data/deine_daten.csv"
# target_col = "label"
# 
# config = TrainerConfig(task="classification", model_type="mlp")
# data_manager = DataManager(config)
# train_loader, test_loader = data_manager.load_csv(csv_path, target_col)
# 
# # Preview
# df = pd.read_csv(csv_path)
# display(df.head())
# df.describe()

In [ ]:
# Daten visualisieren
import torch
X_train = torch.cat([b[0] for b in train_loader]).numpy()
y_train = torch.cat([b[1] for b in train_loader]).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].hist(y_train, bins=20, color='#8b5cf6')
axes[0].set_title("Label Verteilung")
axes[1].boxplot(X_train[:, :4] if X_train.shape[1]>=4 else X_train)
axes[1].set_title("Feature Verteilung")
plt.tight_layout()
plt.show()

## 3️⃣ Modell konfigurieren
Hier kannst du alles einstellen:

In [ ]:
config = TrainerConfig(
    model_type="mlp",          # mlp, random_forest, linear, cnn, transformer
    task="classification",     # classification oder regression
    epochs=20,
    batch_size=32,
    lr=0.001,
    optimizer="adam",          # adam, adamw, sgd
    hidden_dims=[128, 64, 32],  # Neuronen pro Layer
    dropout=0.2,
    activation="relu"          # relu, gelu, tanh
)

print("Konfiguration:")
for k,v in config.__dict__.items():
    print(f"  {k}: {v}")

# Modell Vorschau bauen
trainer = KiTrainer(config)
trainer.build_model(data_manager.input_dim, data_manager.output_dim)
print(f"\n{trainer.model}")

## 4️⃣ Training starten 🚀

In [ ]:
# Training - hier passiert die Magie
model = trainer.train(train_loader, test_loader, data_manager.input_dim, data_manager.output_dim)

print("\n✅ Training fertig!")
print(f"Beste Val Accuracy: {max(trainer.history['val_acc']):.4f}" if trainer.history['val_acc'] else "")

In [ ]:
# Live Plots (auch während Training verfügbar)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

# Loss
ax1.plot(trainer.history['train_loss'], label='Train', color='#06b6d4', linewidth=2)
ax1.plot(trainer.history['val_loss'], label='Val', color='#8b5cf6', linewidth=2)
ax1.set_title("Loss Verlauf", fontsize=14, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.legend()
ax1.grid(True, alpha=0.2)

# Accuracy
if trainer.history['train_acc']:
    ax2.plot(trainer.history['train_acc'], label='Train', color='#10b981', linewidth=2)
    ax2.plot(trainer.history['val_acc'], label='Val', color='#f59e0b', linewidth=2)
    ax2.set_title("Accuracy Verlauf", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Epoch")
    ax2.legend()
    ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("notebook_training_history.png", dpi=150)
plt.show()

## 5️⃣ Modell testen & Vorhersagen

In [ ]:
# Einzelne Vorhersage testen
# Nimm ein Sample aus dem Test Set
import torch
test_batch = next(iter(test_loader))
sample_X, sample_y = test_batch[0][:5], test_batch[1][:5]

trainer.model.eval()
with torch.no_grad():
    preds = trainer.model(sample_X.to(trainer.device))
    if config.task == "classification":
        probs = torch.softmax(preds, dim=1)
        pred_classes = probs.argmax(dim=1).cpu().numpy()
        print("Vorhersagen vs Wahr:")
        for i in range(5):
            print(f"  Sample {i}: Pred={pred_classes[i]} (conf={probs[i].max():.2f}) | True={sample_y[i]}")
    else:
        print(f"Vorhersagen: {preds.cpu().numpy().flatten()}")
        print(f"Wahr: {sample_y.numpy()}")

In [ ]:
# Eigene Werte vorhersagen (z.B. Iris: 4 Features)
def predict_custom(values):
    """values = Liste wie [5.1, 3.5, 1.4, 0.2]"""
    arr = np.array([values])
    arr_scaled = data_manager.scaler.transform(arr)
    result = trainer.predict(arr_scaled)
    if config.task == "classification":
        pred_class = result.argmax()
        confidence = result.max()
        print(f"🔮 Vorhersage: Klasse {pred_class} mit {confidence*100:.1f}% Sicherheit")
        print(f"Wahrscheinlichkeiten: {result}")
        return pred_class
    else:
        print(f"🔮 Vorhersage: {result[0][0]:.4f}")
        return result[0][0]

# Beispiel:
predict_custom([5.1, 3.5, 1.4, 0.2])

## 6️⃣ Export für Produktion

In [ ]:
# PyTorch Modell speichern
import torch
torch.save(trainer.model.state_dict(), "models/mein_modell.pt")
print("✅ Gespeichert: models/mein_modell.pt")

# ONNX Export (für Web, Mobile, etc.)
trainer.export_onnx(data_manager.input_dim, "models/mein_modell.onnx")

# Komplettes Modell mit Scaler für Produktion
import pickle
production_package = {
    'model_state': trainer.model.state_dict() if isinstance(trainer.model, torch.nn.Module) else trainer.model,
    'scaler': data_manager.scaler,
    'config': config,
    'input_dim': data_manager.input_dim,
    'output_dim': data_manager.output_dim
}
with open("models/production_package.pkl", "wb") as f:
    pickle.dump(production_package, f)
print("✅ Production Package: models/production_package.pkl")

## 7️⃣ Bonus: AutoML - Bestes Modell finden

In [ ]:
# Teste automatisch mehrere Modelle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train = torch.cat([b[0] for b in train_loader]).numpy()
y_train = torch.cat([b[1] for b in train_loader]).numpy()
X_test = torch.cat([b[0] for b in test_loader]).numpy()
y_test = torch.cat([b[1] for b in test_loader]).numpy()

results = {}

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
results['RandomForest'] = accuracy_score(y_test, rf.predict(X_test))

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
results['LogisticRegression'] = accuracy_score(y_test, lr.predict(X_test))

# MLP (aus aktuellem Trainer)
if trainer.history['val_acc']:
    results['MLP'] = max(trainer.history['val_acc'])

# Ergebnis
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:20s}: {acc:.4f} {'<- BEST' if acc==max(results.values()) else ''}")

# Plot
plt.figure(figsize=(8,4))
plt.bar(results.keys(), results.values(), color=['#8b5cf6', '#06b6d4', '#10b981'])
plt.title("Modell Vergleich")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

---
## 🎉 Fertig!
Du hast jetzt ein trainiertes Modell. Nächste Schritte:
- Nutze `predict_custom()` für neue Daten
- Lade das Modell in `api_server.py` für eine API
- Exportiere als ONNX für Web/Mobile

Fragen? Einfach im Chat fragen!